# 表单与文件处理

学习目标：接收普通表单和小型文本文件，区分请求体编码，检查上传约束与资源关闭，并通过浏览器提交和下载文件。

前置知识：HTML 表单、HTTP 请求体、Python 文件读写、async/await 与异常处理。

运行环境：Python 3.12、FastAPI 0.141.1 与 python-multipart 0.0.32；表单解析需要 python-multipart。其余依赖和安装步骤见环境入口。

环境准备：[FastAPI 环境与运行说明](README.md)。选择课程环境的 Python 3 (ipykernel)，从空内核顺序运行。Notebook 和终端工作目录均为 content/Web与应用开发/FastAPI。前六节在应用内调用接口；最后一节使用本地端口 8140。

配套脚本：位于 scripts/14-forms-and-files/，用于最后的浏览器操作。

1. [app.py](scripts/14-forms-and-files/app.py)：组合本章的表单、上传、下载与页面路由，供 Uvicorn 导入。
2. [forms.html](scripts/14-forms-and-files/forms.html)：原生 HTML 表单页面。
3. [example.txt](scripts/14-forms-and-files/example.txt)：可上传和下载的 UTF-8 小文本；第 6 节也直接读取此文件。

## 1 用 Form 接收一个标题

先写一个只接收标题的接口。Form 告诉 FastAPI 从表单字段中读取 title；这里仍可用 min_length 和 max_length 限制字符串长度。字段没有默认值，因此必须提供。

TestClient 在应用内调用接口，不监听网络端口。它的 data 参数把字典编码为普通表单；with 语句在调用结束后关闭客户端。

In [1]:
from typing import Annotated

from fastapi import FastAPI, Form
from fastapi.testclient import TestClient

app = FastAPI()

@app.post("/form-note")
def form_note(title: Annotated[str, Form(min_length=1, max_length=30)]) -> dict[str, str]:
    return {"title": title}

with TestClient(app) as client:
    response = client.post("/form-note", data={"title": "学习表单"})

# title 来自请求体中的表单字段，响应仍然可以是 JSON。
assert response.status_code == 200
assert response.json() == {"title": "学习表单"}
print(response.request.headers["content-type"])  # 预期：application/x-www-form-urlencoded。
print(response.status_code, response.json())  # 预期：200 {'title': '学习表单'}。

application/x-www-form-urlencoded
200 {'title': '学习表单'}


C:\Users\ZHUANG\miniconda3\envs\hands-on-computing\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


## 2 相同字段不代表相同请求体编码

Content-Type 声明请求体采用什么编码。下面三种编码不能因为都包含 title 就互换。

| 编码名称 | 中文名称／含义 | 本章的发送方式 |
| --- | --- | --- |
| application/x-www-form-urlencoded | 普通表单编码，字段按键值对编码 | data 传字典 |
| multipart/form-data | 多部分表单编码，普通字段和文件分成多个部分 | data 与 files 配合 |
| application/json | JSON 编码，请求体是一个 JSON 值 | json 传字典 |

向刚才的 Form 接口发送 JSON，并不会把 JSON 字段自动变成表单字段。同一个请求体不能同时以普通 JSON 和 multipart 作为外层编码；需要文件和普通字段时，可以把它们放在同一个 multipart 请求中。

In [2]:
with TestClient(app) as client:
    wrong_encoding = client.post("/form-note", json={"title": "学习表单"})
    missing_title = client.post("/form-note", data={"other": "没有标题"})
    long_title = client.post("/form-note", data={"title": "字" * 31})

print(wrong_encoding.request.headers["content-type"])  # 预期：application/json。
for label, response in [
    ("JSON 发给 Form", wrong_encoding),
    ("缺少标题", missing_title),
    ("标题过长", long_title),
]:
    # 这三次都触发请求校验；错误位置仍指向请求体中的 title。
    assert response.status_code == 422
    error = response.json()["detail"][0]
    # 预期：三项均为 422，位置为 body/title；错误依次为 missing、missing、string_too_long。
    print(label, response.status_code, error["loc"], error["type"])

application/json
JSON 发给 Form 422 ['body', 'title'] missing
缺少标题 422 ['body', 'title'] missing
标题过长 422 ['body', 'title'] string_too_long


## 3 读取、回到开头和关闭 UploadFile

UploadFile 表示上传文件。FastAPI 接收上传时，底层文件先在内存中缓冲，超过缓冲阈值后可转到磁盘；使用 UploadFile 不等于上传内容始终都在内存中。

filename 是客户端提供的文件名，content_type 是客户端声明的媒体类型，它们不能证明文件实际内容。file 则是可读写的底层文件对象。

在异步函数中，await upload.read(n) 最多读取 n 个字节，n 是本次读取上限；await upload.seek(0) 回到开头；await upload.close() 关闭文件。下面先用 BytesIO 在内存中构造一个小文件，只观察读取位置和关闭，不发送 HTTP 请求。

In [3]:
from io import BytesIO

from fastapi import UploadFile

sample = UploadFile(filename="note.txt", file=BytesIO(b"FastAPI"))
try:
    first = await sample.read(3)
    rest = await sample.read(4)
    await sample.seek(0)
    again = await sample.read(7)
finally:
    await sample.close()

# 连续读取会推进位置；seek(0) 后可以重新从开头读取。
assert (first, rest, again) == (b"Fas", b"tAPI", b"FastAPI")
assert sample.file.closed
print(first, rest, again)  # 预期：b'Fas' b'tAPI' b'FastAPI'。
print("底层文件已关闭：", sample.file.closed)  # 预期：底层文件已关闭： True。

b'Fas' b'tAPI' b'FastAPI'
底层文件已关闭： True


## 4 在 multipart 中接收并限制小文本

给 app 增加一个带标题的上传接口。title 使用 Form，upload 标注为 UploadFile；这两个字段来自同一个 multipart 请求。下面的约束由本示例自行规定：文件名以 .txt 结尾、文件内容不超过 1024 字节、内容能按 UTF-8 严格解码。

read(MAX_BYTES + 1) 多读一个字节，用来区别“恰好达到上限”和“超过上限”。这里限制的是路由从已解析文件中读取和处理的数据；文件此前可能已传输并被暂存，因此它不是整个 HTTP 请求的传输上限。

文件名只用于检查扩展名和返回信息，不参与磁盘路径拼接。扩展名是一项命名约定，实际内容仍需解码检查；这个小例子不把后缀或 content_type 当作内容可信的证明。finally 覆盖路由内的返回和异常路径。

In [4]:
from fastapi import HTTPException

MAX_BYTES = 1024

@app.post("/upload-note")
async def upload_note(
    title: Annotated[str, Form(min_length=1, max_length=30)],
    upload: UploadFile,
) -> dict[str, str | int]:
    # 路由接管上传文件后，所有返回或异常路径都必须进入 finally 关闭。
    try:
        if not upload.filename or not upload.filename.lower().endswith(".txt"):
            raise HTTPException(400, "只接受 .txt 文件")
        # 比允许上限多读 1 字节，才可判断文件是否超限，而不必读完整文件。
        data = await upload.read(MAX_BYTES + 1)
        if len(data) > MAX_BYTES:
            raise HTTPException(413, "文件不能超过 1024 字节")
        # 解码失败转换为本接口约定的 422；其他错误保持原传播方式。
        try:
            text = data.decode("utf-8")
        except UnicodeDecodeError as error:
            raise HTTPException(422, "文件必须采用 UTF-8 编码") from error
        return {
            "title": title,
            "filename": upload.filename,
            "bytes": len(data),
            "text": text,
        }
    finally:
        # 路由接管文件以后，无论返回还是抛出异常，都关闭文件。
        await upload.close()

继续用同一个 app。客户端的 files 字典以 upload 为字段名；三元组依次给出文件名、文件内容和媒体类型。data 中的 title 作为普通字段一同发送。HTTPX 负责生成 multipart 的分隔标记和 Content-Type，无需手工填写这个请求头。

In [5]:
text = "FastAPI 表单与文件\n"
payload = text.encode("utf-8")
with TestClient(app) as client:
    response = client.post(
        "/upload-note",
        data={"title": "文件练习"},
        files={"upload": ("note.txt", payload, "text/plain")},
    )

assert response.status_code == 200
assert response.json()["text"] == text
assert response.json()["bytes"] == len(payload)
# 只展示外层编码名；具体 multipart 分隔标记由客户端生成。
print(response.request.headers["content-type"].split(";")[0])  # 预期：multipart/form-data。
# 预期：200 {'title': '文件练习', 'filename': 'note.txt', 'bytes': 24, 'text': 'FastAPI 表单与文件\n'}。
print(response.status_code, response.json())

multipart/form-data
200 {'title': '文件练习', 'filename': 'note.txt', 'bytes': 24, 'text': 'FastAPI 表单与文件\n'}


## 5 拒绝条件和关闭路径

先通过 HTTP 请求检查三种应用约束，再检查缺失文件的请求校验。400、413 和文件编码错误时的 422，是这个接口为这些情况选择的响应；不是 UploadFile 自动附加的文件约束。

In [6]:
cases = [
    ("错误后缀", "note.bin", b"hello", 400),
    ("超过字节上限", "note.txt", b"a" * 1025, 413),
    ("不是 UTF-8", "note.txt", b"\xff", 422),
]
with TestClient(app) as client:
    for label, filename, body, expected in cases:
        response = client.post(
            "/upload-note", data={"title": "拒绝检查"},
            files={"upload": (filename, body, "text/plain")},
        )
        assert response.status_code == expected
        print(label, response.status_code, response.json())  # 预期：错误后缀、超出上限、非 UTF-8 依次为 400、413、422，detail 给出对应原因。
    missing = client.post("/upload-note", data={"title": "没有文件"})
    assert missing.status_code == 422
    print("缺少文件", missing.status_code, missing.json()["detail"][0]["loc"])  # 预期：422 ['body', 'upload']。

错误后缀 400 {'detail': '只接受 .txt 文件'}
超过字节上限 413 {'detail': '文件不能超过 1024 字节'}
不是 UTF-8 422 {'detail': '文件必须采用 UTF-8 编码'}
缺少文件 422 ['body', 'upload']


字节数和字符数是不同的计量。约束针对编码后的文件字节，因此要同时检查恰好 1024 字节与超过 1024 字节的情况。

In [7]:
with TestClient(app) as client:
    boundary = client.post(
        "/upload-note", data={"title": "边界检查"},
        files={"upload": ("note.txt", b"a" * 1024, "text/plain")},
    )
    chinese = client.post(
        "/upload-note", data={"title": "字节与字符"},
        files={"upload": ("note.txt", ("字" * 400).encode("utf-8"), "text/plain")},
    )

assert boundary.status_code == 200 and boundary.json()["bytes"] == 1024
assert chinese.status_code == 413
print("1024 字节：", boundary.status_code)  # 预期：1024 字节： 200。
print("400 个汉字的 UTF-8 字节数：", len(("字" * 400).encode("utf-8")))  # 预期：400 个汉字的 UTF-8 字节数： 1200。
print("400 个汉字：", chinese.status_code)  # 预期：400 个汉字： 413。

1024 字节：

 200
400 个汉字的 UTF-8 字节数： 1200
400 个汉字： 413


HTTP 响应不能直接告诉我们底层文件是否关闭。下面直接调用已经定义的 upload_note，保留传入文件的引用来检查 closed。这个观察只验证路由自身的清理逻辑，绕过了 FastAPI 的参数提取和校验；HTTP 行为已经由前面的请求检查。

In [8]:
cleanup_cases = [("成功", "note.txt", b"hello", 200), *cases]
for label, filename, body, expected in cleanup_cases:
    uploaded = UploadFile(filename=filename, file=BytesIO(body))
    try:
        result = await upload_note(title="关闭检查", upload=uploaded)
    except HTTPException as error:
        observed = error.status_code
    else:
        observed = 200
    assert observed == expected
    assert uploaded.file.closed
    print(label, "底层文件已关闭：", uploaded.file.closed)  # 预期：成功和三种失败分支的底层文件已关闭均为 True。

成功 底层文件已关闭： True
错误后缀 底层文件已关闭： True
超过字节上限 底层文件已关闭： True
不是 UTF-8 底层文件已关闭： True


## 6 用 FileResponse 下载固定文件

FileResponse 根据给定路径发送文件内容。path 决定读取哪个文件；filename 决定 Content-Disposition 中建议使用的下载名，默认采用 attachment，也就是作为附件下载。

本例始终提供配套的 example.txt，路径相对于 Notebook 工作目录。下载端点没有接收用户提供的路径或文件名；文件在响应发送期间保持存在。

In [9]:
from pathlib import Path

from fastapi.responses import FileResponse

sample_path = Path("scripts/14-forms-and-files/example.txt")


@app.get("/example/download")
def download_example() -> FileResponse:
    return FileResponse(sample_path, media_type="text/plain", filename="example.txt")

客户端的 content 保存响应字节；响应头 Content-Disposition 则告诉客户端如何处理这份内容。分别核对它们，避免只看到 200 就认为下载内容正确。

In [10]:
with TestClient(app) as client:
    response = client.get("/example/download")

assert response.status_code == 200
assert response.content == sample_path.read_bytes()
assert response.headers["content-disposition"] == 'attachment; filename="example.txt"'
print(response.headers["content-type"])  # 预期：text/plain; charset=utf-8。
print(response.headers["content-disposition"])  # 预期：attachment; filename="example.txt"。
print(response.content.decode("utf-8"), end="")  # 预期：FastAPI 表单与文件。

text/plain; charset=utf-8
attachment; filename="example.txt"
FastAPI 表单与文件


## 7 用原生 HTML 表单提交和下载

浏览器可以直接提交表单，不需要 JavaScript。form 的 action 指定接口，method 指定请求方法；input 的 name 要与接口参数一致。label 的 for 对应输入框的 id，为控件提供可见名称。

普通标题表单采用默认的 application/x-www-form-urlencoded；含文件的表单必须指定 enctype="multipart/form-data"。上传部分的核心 HTML 如下，完整页面同时包含普通标题表单和下载链接。

```html
<form action="/upload-note" method="post" enctype="multipart/form-data">
  <label for="upload-title">文件标题</label>
  <input id="upload-title" name="title" maxlength="30" required>
  <label for="upload-file">UTF-8 文本文件</label>
  <input id="upload-file" type="file" name="upload" accept=".txt,text/plain" required>
  <button type="submit">上传并读取</button>
</form>
```

accept 帮助文件选择器提示可选文件，required 和 maxlength 改善填写体验；请求仍须接受服务端检查。页面与 API 由同一应用提供，两个 action 都使用站点内的绝对路径。

把固定的 HTML 文件交给 FileResponse 即可显示页面。这里不传 filename，避免把页面作为下载附件。下面继续给 Notebook 中的 app 添加首页，并检查返回内容。

In [11]:
page_path = Path("scripts/14-forms-and-files/forms.html")


@app.get("/", include_in_schema=False)
def form_page() -> FileResponse:
    return FileResponse(page_path, media_type="text/html")


with TestClient(app) as client:
    page = client.get("/")

assert page.status_code == 200
assert 'enctype="multipart/form-data"' in page.text
assert "content-disposition" not in page.headers
print(page.status_code, page.headers["content-type"])  # 预期：200 text/html; charset=utf-8。

200 text/html; charset=utf-8


浏览器要访问真实端口，因此配套 app.py 组合以上四条路由，HTML 文件只提供表单。脚本中使用 Path(\_\_file\_\_).resolve().parent 得到自身目录，再定位同目录的 forms.html 和 example.txt；导入 app.py 不会启动服务器。

Step 1：在课程目录的终端启动本地服务。

```powershell
python -m uvicorn app:app --app-dir scripts/14-forms-and-files --host 127.0.0.1 --port 8140
```

Step 2：等终端显示应用启动完成后，在浏览器打开 http://127.0.0.1:8140/。

Step 3：在“普通表单”的“标题”中填写“学习表单”，点击“提交标题”，查看返回的 title；返回首页。

Step 4：在“上传小文本”的“文件标题”中填写“文件练习”，选择配套 example.txt，点击“上传并读取”，检查 JSON 中的 title、filename、bytes 和 text 与输入一致；返回首页。

![浏览器中已填写文件标题并选中 example.txt 的上传表单](image/14-upload-form.png)

图中已填写“文件练习”并选中 example.txt，点击“上传并读取”后会显示接口返回的 JSON。

Step 5：在本地准备一个超过 1024 字节的 UTF-8 .txt 文件，例如含 1025 个英文字母，上传后应收到“文件不能超过 1024 字节”的错误。浏览器开发者工具的 Network（网络）面板可查看本次请求的 413 状态。

Step 6：返回首页，点击“下载示例文本”，在浏览器下载列表中找到 example.txt，打开后核对文本内容。

Step 7：回到服务终端按 Ctrl+C，等待应用关闭完成；再次刷新首页，确认端口已不能访问。

上传内容只在路由中读取并返回，没有另外保存为业务文件。练习产生的本地文本和下载副本在查看后可删除，课程配套 example.txt 保留。

## 本章小结

（1）Form、JSON 和 multipart 对应不同请求体编码；响应采用 JSON 不会改变请求体的编码要求。

（2）UploadFile 支持有限读取、重新定位与关闭；文件名和媒体类型只是客户端提供的信息。

（3）示例分别检查后缀、字节数与 UTF-8 解码，并用 finally 关闭路由接管的文件；有限读取不等于限制整个请求传输。

（4）FileResponse 的文件路径与下载名职责不同。原生 HTML 表单可以调用同一接口，浏览器输入限制仍需服务端校验配合。

自查：为什么 400 个汉字可能超过 1024 字节？为什么一个带 .txt 后缀的文件仍可能无法按 UTF-8 解码？

## 练习

1. 给普通表单增加可选的 category 字段，默认值为“未分类”。分别发送缺失字段和自定义字段的请求，确认响应中的分类符合输入。

In [ ]:
# 在此完成本题，按题目逐项核对接口与资源结果。

2. 将小文本上限改为 2048 字节，同时修改错误信息。用 2048 和 2049 字节分别请求，核对成功与 413；浏览器表单提示也要同步。

In [ ]:
# 在此完成本题，按题目逐项核对接口与资源结果。

3. 为上传接口增加“不能是空文件”的约束。发送空的 .txt 文件和一个只含换行的 .txt 文件，明确自己的规则并断言响应；直接调用处理函数确认两条路径都关闭文件。

In [ ]:
# 在此完成本题，按题目逐项核对接口与资源结果。

4. 增加一个指向固定说明文件的下载接口，磁盘文件名与下载名分别设置。核对响应字节对应磁盘文件，Content-Disposition 使用指定下载名；接口不接收任意文件路径。

In [ ]:
# 在此完成本题，按题目逐项核对接口与资源结果。

### 第 3 题提示与解析

提示 1：先明确本题采用“零字节为空”，还是“解码后去掉空白为空”；两者对换行文件的判断不同。

提示 2：在已有 `try/finally` 内增加该规则，用 400 表示本例选择的空文件拒绝；直接调用时检查 `uploaded.file.closed`。

解析：若按零字节定义，`len(data) == 0` 时拒绝，空文件为 400，换行文件为 200；若按去空白后的文本定义，`not text.strip()` 时拒绝，两种文件均为 400。两种规则都应在退出时关闭文件，不能用状态码代替关闭状态检查。

其他提示：分类字段仍用 `Form`；调整大小上限时同步提示并继续多读 1 字节；下载的 `path` 与 `filename` 分别决定内容和建议文件名。

## 参考与引用来源

- **FastAPI 官方文档**：[Form Data](https://fastapi.tiangolo.com/tutorial/request-forms/)，定位 Import Form、Define Form parameters、About Form Fields，支持显式表单声明、校验与 JSON 编码边界；[Request Files](https://fastapi.tiangolo.com/tutorial/request-files/)，定位 UploadFile、UploadFile attributes 与 async methods，支持缓冲文件、元数据、读取、定位和关闭；[UploadFile 参考](https://fastapi.tiangolo.com/reference/uploadfile/)，定位构造参数、read、seek、close；[Request Forms and Files](https://fastapi.tiangolo.com/tutorial/request-forms-and-files/)，支持同一 multipart 请求的普通字段与文件；[Handling Errors](https://fastapi.tiangolo.com/tutorial/handling-errors/#use-httpexception)，支持以 HTTPException 返回预期错误；[Testing](https://fastapi.tiangolo.com/tutorial/testing/#using-testclient)，支持应用内接口调用。
- **Starlette 官方文档**：[FileResponse](https://starlette.dev/responses/#fileresponse)，定位 path、filename、content_disposition_type 与文件响应头；[Request files](https://starlette.dev/requests/#request-files)，定位文件对象、文件元数据及异步方法。文件内容在解析后的文件对象中读取，示例的读取上限不能代表整个请求的传输上限。
- **HTTPX 官方文档**：[QuickStart](https://www.python-httpx.org/quickstart/)，定位 Sending Form Encoded Data、Sending Multipart File Uploads、Sending JSON Encoded Data、Binary Response Content 和 Response Headers，支持 data、files 三元组、json 及响应检查。
- **Python 3.12 官方文档**：[BytesIO](https://docs.python.org/3.12/library/io.html#io.BytesIO) 与 [IOBase.closed](https://docs.python.org/3.12/library/io.html#io.IOBase.closed)，支持内存字节流与关闭状态；[bytes.decode](https://docs.python.org/3.12/library/stdtypes.html#bytes.decode)，支持 UTF-8 严格解码；[str.strip](https://docs.python.org/3.12/library/stdtypes.html#str.strip)，支持练习中去除两端空白的规则；[Defining Clean-up Actions](https://docs.python.org/3.12/tutorial/errors.html#defining-clean-up-actions)，支持 finally 在返回和异常时执行；[pathlib](https://docs.python.org/3.12/library/pathlib.html)，定位 Path.resolve、PurePath.parent 和 Path.read_bytes，支持资源定位与文件字节检查。
- **WHATWG HTML Living Standard**：[配置表单与服务器通信](https://html.spec.whatwg.org/multipage/forms.html#configuring-a-form-to-communicate-with-a-server)，支持 action、method、name 与 enctype；[File Upload state](https://html.spec.whatwg.org/multipage/input.html#file-upload-state-(type=file))，定位 accept 属性及文件数据仍需验证的边界。
- **W3C WAI**：[Labeling Controls](https://www.w3.org/WAI/tutorials/forms/labels/#associating-labels-explicitly)，支持 label 的 for 与控件 id 显式关联。
- **Uvicorn 官方文档**：[Settings](https://uvicorn.dev/settings/)，定位 Application 与 Socket Binding，支持 app 导入字符串、app-dir、本地地址与端口选项。